# Team EDA

Compare team win rates, offense, defense, home/away splits, season results, consistency and chronological streaks.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'backend' / 'app').is_dir())
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None
import pandas as pd
import plotly.express as px
from backend.app import eda

DATA_DIRS = [ROOT / 'data' / 'processed', ROOT / 'data' / 'raw', ROOT / 'Raw']
teams_result = eda.teams_analysis(DATA_DIRS)
if teams_result['available']:
    teams = pd.DataFrame(teams_result['teams'])
    display(teams.head(30))
    print('Unavailable analyses:', teams_result['unavailable_analyses'])
else:
    print(teams_result['reason'])

In [ ]:
if teams_result.get('chart_data', {}).get('teams'):
    px.bar(teams_result['chart_data'], x='teams', y='win_percentage',
           title='Team win percentage', labels={'teams': 'Team', 'win_percentage': 'Win %'}).show()
    comparison = teams[['team', 'average_points_scored', 'average_points_conceded']].melt(
        id_vars='team', var_name='measure', value_name='points')
    px.bar(comparison, x='team', y='points', color='measure', barmode='group',
           title='Team offense and points conceded').show()
    px.scatter(teams, x='average_points_scored', y='average_points_conceded',
               size='games', color='win_percentage', hover_name='team',
               title='Scoring vs points allowed').show()

In [ ]:
season = pd.DataFrame(teams_result.get('season_performance', []))
if not season.empty:
    candidates = season.groupby('team')['games'].sum().nlargest(8).index
    px.line(season[season['team'].isin(candidates)], x='season', y='win_percentage',
            color='team', markers=True, title='Season-by-season team win percentage').show()
else:
    print('Season-by-season comparison unavailable.')
if teams_result.get('available'):
    display(teams[['team', 'home_win_percentage', 'away_win_percentage']].head(30))
else:
    print('Home/away team comparison unavailable.')